# Assignment 1.2 — RAG-Based Intelligent Document Assistant (LangChain + LlamaIndex)

**Objective:** a document question-answering system that
1. **ingests** documents (PDF, text, markdown and web pages),
2. **indexes** them with **LlamaIndex** (`VectorStoreIndex`, persisted to `./storage`),
3. performs **semantic retrieval** with embeddings (HuggingFace `all-MiniLM-L6-v2`),
4. uses **LangChain** for LLM orchestration and response generation,
5. answers with **context-aware, cited** responses (and remembers the conversation),
6. supports **multi-document querying, comparison and summarisation**.


## Step 0 — Setup

In [ ]:
import sys
print(sys.version)
import llama_index.core, langchain_core
print("llama-index-core", llama_index.core.__version__, "| langchain-core", langchain_core.__version__)
print("Everything is okay!!!")

## Step 1 — Configuration
API keys from `.env`, the LangChain LLM, and LlamaIndex global `Settings` (embedding model + chunking).

In [ ]:
"""
Configuration: API keys, LangChain LLM, and LlamaIndex global Settings.

"""
import os
import re
import warnings
from pathlib import Path

# Mac (Apple Silicon) safety flags: avoid OpenMP clash + tokenizer fork warnings
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from dotenv import load_dotenv  # noqa: E402

warnings.filterwarnings("ignore")
load_dotenv()
# Hugging Face token (free) -> faster, rate-limit-free download of the embedding model
if os.getenv("HUGGINGFACE_API_KEY") and not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = os.getenv("HUGGINGFACE_API_KEY")

try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:                       # running inside the notebook
    PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "documents"        # put your PDFs / txt / md files here
PERSIST_DIR = PROJECT_DIR / "storage"                 # LlamaIndex saves the vector index here

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq").lower()        # groq | gemini (free keys) | ollama (free, local)
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3.5:2b")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "512"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "64"))


def get_llm(temperature: float = 0.1):
    """LangChain chat model - LangChain does ALL the LLM work in this project."""
    if LLM_PROVIDER == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)

    if LLM_PROVIDER == "gemini":
     
        from langchain_google_genai import ChatGoogleGenerativeAI
        key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
        if not key:
            raise EnvironmentError("GEMINI_API_KEY is missing. Add it to your .env file.")
        return ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=temperature, google_api_key=key, max_retries=3)

    from langchain_groq import ChatGroq
    if not os.getenv("GROQ_API_KEY"):
        raise EnvironmentError("GROQ_API_KEY is missing. Add it to your .env file.")
    kwargs = dict(model=GROQ_MODEL, temperature=temperature, max_retries=5)
    if "gpt-oss" in GROQ_MODEL:
        kwargs["reasoning_effort"] = "low"
    return ChatGroq(**kwargs)


def configure_llamaindex(embed_model=None):
    """
    LlamaIndex is used ONLY for indexing + retrieval:
      - embeddings: local HuggingFace MiniLM (free, ~90 MB, fast on M1 CPU)
      - llm: disabled (None) because answer generation is done by LangChain
    """
    from llama_index.core import Settings
    from llama_index.core.node_parser import SentenceSplitter

    if embed_model is None:
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL, device="cpu")
    Settings.embed_model = embed_model
    Settings.llm = None
    Settings.node_parser = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    return Settings


def strip_think(text) -> str:
    """Remove <think>...</think> blocks that some local reasoning models output."""
    text = text if isinstance(text, str) else getattr(text, "content", str(text))
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def provider_label() -> str:
    model = {"groq": GROQ_MODEL, "gemini": GEMINI_MODEL, "ollama": OLLAMA_MODEL}.get(LLM_PROVIDER, GROQ_MODEL)
    return f"{LLM_PROVIDER}:{model}"

In [ ]:
llm = get_llm()
print("LLM:", provider_label())
print(llm.invoke("Say 'ready' in one word").content)

configure_llamaindex()          # downloads the ~90 MB embedding model the first time
print("Embedding model ready")

## Step 2 — LlamaIndex: ingestion, indexing, persistence and retrieval

In [ ]:
"""
LlamaIndex part: document ingestion, indexing, persistence and retrieval.

- SimpleDirectoryReader                 -> load PDFs / text files
- VectorStoreIndex.from_documents       -> chunk + embed + index
- StorageContext / load_index_from_storage -> persist index to ./storage and reload
- VectorIndexRetriever                  -> top-k semantic retrieval
- SimilarityPostprocessor               -> drop chunks below a similarity cutoff
"""
import re
from pathlib import Path

import requests
from llama_index.core import Document, SimpleDirectoryReader, StorageContext, VectorStoreIndex, load_index_from_storage
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.vector_stores import FilterOperator, MetadataFilter, MetadataFilters


SUPPORTED_EXTS = [".pdf", ".txt", ".md"]


# ------------------------------------------------------------------ Ingestion
def load_documents_from_folder(folder=DATA_DIR) -> list:
    folder = Path(folder)
    if not folder.exists() or not any(p.suffix.lower() in SUPPORTED_EXTS for p in folder.iterdir()):
        return []
    reader = SimpleDirectoryReader(input_dir=str(folder), required_exts=SUPPORTED_EXTS, recursive=True)
    return reader.load_data(show_progress=True)


def load_files(paths: list) -> list:
    return SimpleDirectoryReader(input_files=[str(p) for p in paths]).load_data()


def load_web_page(url: str) -> list:
    """Fetch a web page and turn it into a LlamaIndex Document (web data ingestion)."""
    from bs4 import BeautifulSoup

    html = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"}).text
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
        tag.decompose()
    title = (soup.title.string or url).strip() if soup.title else url
    text = re.sub(r"\n\s*\n+", "\n\n", soup.get_text("\n")).strip()
    name = re.sub(r"[^A-Za-z0-9]+", "_", title)[:60] + ".web"
    return [Document(text=text, metadata={"file_name": name, "source_url": url, "title": title})]


# ------------------------------------------------------------------ Index
def build_index(documents: list, persist_dir=PERSIST_DIR) -> VectorStoreIndex:
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=str(persist_dir))
    return index


def load_or_build_index(data_dir=DATA_DIR, persist_dir=PERSIST_DIR, rebuild: bool = False):
    """Load the saved index if it exists, otherwise build it from the documents folder."""
    persist_dir = Path(persist_dir)
    if persist_dir.exists() and any(persist_dir.iterdir()) and not rebuild:
        storage_context = StorageContext.from_defaults(persist_dir=str(persist_dir))
        return load_index_from_storage(storage_context)
    docs = load_documents_from_folder(data_dir)
    if not docs:
        index = VectorStoreIndex.from_documents([])      # empty index; documents can be added later
        index.storage_context.persist(persist_dir=str(persist_dir))
        return index
    return build_index(docs, persist_dir)


def add_documents(index: VectorStoreIndex, documents: list, persist_dir=PERSIST_DIR) -> int:
    """Insert new documents into an existing index and save it."""
    for d in documents:
        index.insert(d)
    index.storage_context.persist(persist_dir=str(persist_dir))
    return len(documents)


def list_documents(index: VectorStoreIndex) -> list:
    """Unique file names in the index (PDFs are stored one Document per page)."""
    names = {info.metadata.get("file_name", "unknown") for info in index.ref_doc_info.values()}
    return sorted(names)


def get_document_nodes(index: VectorStoreIndex, file_name: str) -> list:
    """All chunks (nodes) of one document, in reading order."""
    nodes = [n for n in index.docstore.docs.values() if n.metadata.get("file_name") == file_name]

    def order(n):
        page = str(n.metadata.get("page_label", "0"))
        return (int(page) if page.isdigit() else 0, n.start_char_idx or 0)

    return sorted(nodes, key=order)


# ------------------------------------------------------------------ Retrieval
def make_filters(file_names: list | None):
    if not file_names:
        return None
    return MetadataFilters(filters=[MetadataFilter(key="file_name", value=list(file_names), operator=FilterOperator.IN)])


def retrieve(index: VectorStoreIndex, question: str, top_k: int = 4, file_names: list | None = None,
             similarity_cutoff: float = 0.25) -> list:
    """Semantic retrieval with LlamaIndex: VectorIndexRetriever + SimilarityPostprocessor."""
    retriever = VectorIndexRetriever(index=index, similarity_top_k=top_k, filters=make_filters(file_names))
    nodes = retriever.retrieve(question)
    kept = SimilarityPostprocessor(similarity_cutoff=similarity_cutoff).postprocess_nodes(nodes)
    return kept or nodes[:2]           # never return nothing if the index has data


def query_engine_preview(index: VectorStoreIndex, question: str, top_k: int = 4):
    
    engine = index.as_query_engine(similarity_top_k=top_k, response_mode="no_text")
    return engine.query(question)

In [ ]:
# 2.1 Load documents (one LlamaIndex Document per PDF page)
docs = load_documents_from_folder(DATA_DIR)
print(len(docs), "pages loaded")
docs[0].metadata

In [ ]:
# 2.2 Build the index and persist it to ./storage (re-runs will just load it)
index = load_or_build_index(DATA_DIR, PERSIST_DIR)
print("Documents in index:", list_documents(index))
print("Chunks (nodes):", len(index.docstore.docs))

In [ ]:
# 2.3 Semantic retrieval: VectorIndexRetriever + SimilarityPostprocessor
nodes = retrieve(index, "What is the ReAct prompting approach?", top_k=4)
for n in nodes:
    print(round(n.score, 3), n.node.metadata["file_name"], "page", n.node.metadata.get("page_label"))
    print("   ", n.node.get_content()[:150].replace("\n", " "), "...")

In [ ]:
# 2.4 Multi-document querying: restrict retrieval to selected files with MetadataFilters
nodes = retrieve(index, "What is multi-head attention?", top_k=3, file_names=["Attention_Is_All_You_Need.pdf"])
[(n.node.metadata["file_name"], n.node.metadata.get("page_label")) for n in nodes]

In [ ]:
# 2.5 LlamaIndex query engine (retrieval only, because Settings.llm = None)
resp = query_engine_preview(index, "types of hallucination in LLMs")
for sn in resp.source_nodes:
    print(round(sn.score, 3), sn.node.metadata["file_name"], sn.node.metadata.get("page_label"))

In [ ]:
# 2.6 Web data ingestion - add a web page to the same index
web_docs = load_web_page("https://en.wikipedia.org/wiki/Retrieval-augmented_generation")
add_documents(index, web_docs)
list_documents(index)

## Step 3 — LangChain: prompts, chains, runnables and output parsing
`DocumentAssistant` wraps the LlamaIndex retriever in a `RunnableLambda` and builds LCEL chains for QA (with a Pydantic structured answer), conversation memory, summarisation (map-reduce) and comparison.

In [ ]:
"""
LangChain part: LLM orchestration and response generation on top of the LlamaIndex retriever.

"""
from typing import List, Literal

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from pydantic import BaseModel, Field



# ------------------------------------------------------------------ Output schema
class RAGAnswer(BaseModel):
    answer: str = Field(description="Answer in Markdown, using ONLY the context. Cite sources inline like [1], [2].")
    sources_used: List[int] = Field(description="Numbers of the context chunks actually used, e.g. [1, 3]")
    confidence: Literal["high", "medium", "low"] = Field(
        description="high = clearly answered by context, medium = partially, low = not found in context")
    follow_up_questions: List[str] = Field(description="2 short follow-up questions the user could ask next")


def format_nodes(nodes) -> str:
    """Turn LlamaIndex nodes into a numbered context block with citations."""
    parts = []
    for i, n in enumerate(nodes, 1):
        md = n.node.metadata
        page = f", page {md['page_label']}" if md.get("page_label") else ""
        parts.append(f"[{i}] (source: {md.get('file_name', 'unknown')}{page})\n{n.node.get_content().strip()}")
    return "\n\n".join(parts) if parts else "NO CONTEXT FOUND"


def nodes_to_sources(nodes) -> list:
    return [{
        "n": i,
        "file_name": n.node.metadata.get("file_name", "unknown"),
        "page": n.node.metadata.get("page_label", ""),
        "score": round(float(n.score or 0), 3),
        "text": n.node.get_content().strip(),
    } for i, n in enumerate(nodes, 1)]


class DocumentAssistant:
    """LangChain orchestration around a LlamaIndex index."""

    def __init__(self, index, llm, top_k: int = 4, similarity_cutoff: float = 0.25):
        self.index, self.llm = index, llm
        self.top_k, self.similarity_cutoff = top_k, similarity_cutoff
        self.chat_history = []                # list of HumanMessage / AIMessage
        self._build_chains()

    # -------------------------------------------------------------- chains
    def _build_chains(self):
        to_text = StrOutputParser() | RunnableLambda(strip_think)

        # 1) Condense a follow-up question into a standalone question (conversation memory)
        self.condense_chain = ChatPromptTemplate.from_messages([
            ("system", "Rewrite the user's latest question as a standalone question using the chat history. "
                       "Return ONLY the rewritten question. If it is already standalone, return it unchanged."),
            MessagesPlaceholder("history"),
            ("human", "{question}"),
        ]) | self.llm | to_text

        # 2) Grounded QA with structured output (PydanticOutputParser)
        self.answer_parser = PydanticOutputParser(pydantic_object=RAGAnswer)
        qa_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are an intelligent document assistant. Answer ONLY from the context below. "
             "If the answer is not in the context, say \"I could not find this in the documents.\" and set "
             "confidence to low. Cite chunk numbers like [1]. Be clear and concise.\n\n"
             "Context:\n{context}\n\n{format_instructions}"),
            ("human", "{question}"),
        ]).partial(format_instructions=self.answer_parser.get_format_instructions())

        retriever_runnable = RunnableLambda(lambda x: self._retrieve(x["question"], x.get("file_names")))
        # RunnableParallel = retrieve + pass the question through, same pattern as 3_rag_v1.py
        self.qa_chain = (
            RunnableParallel({
                "nodes": retriever_runnable,
                "question": RunnableLambda(lambda x: x["question"]),
            })
            | RunnableParallel({
                "nodes": RunnableLambda(lambda x: x["nodes"]),
                "answer": (RunnablePassthrough.assign(context=lambda x: format_nodes(x["nodes"]))
                           | qa_prompt | self.llm | to_text),
            })
        )

        # Plain-text fallback prompt (used if the model's JSON cannot be parsed)
        self.plain_qa_chain = ChatPromptTemplate.from_messages([
            ("system", "Answer ONLY from the context. Cite chunk numbers like [1]. If not found, say so.\n\n"
                       "Context:\n{context}"),
            ("human", "{question}"),
        ]) | self.llm | to_text

        # 3) Summarisation - map step and reduce step
        self.map_chain = ChatPromptTemplate.from_template(
            "Summarise the following part of the document '{file_name}' in 4-6 bullet points. "
            "Keep key facts, numbers and definitions.\n\n{text}"
        ) | self.llm | to_text
        self.reduce_chain = ChatPromptTemplate.from_template(
            "You are given partial summaries of the document '{file_name}'. Write a final summary in Markdown with:\n"
            "### Overview (2-3 sentences)\n### Key Points (5-8 bullets)\n### Conclusion (1-2 sentences)\n\n"
            "Partial summaries:\n{summaries}"
        ) | self.llm | to_text

        # 4) Multi-document comparison
        self.compare_chain = ChatPromptTemplate.from_messages([
            ("system", "You compare multiple documents. Use ONLY the context, which is grouped by document. "
                       "Answer the question for each document, then give a short comparison table in Markdown "
                       "and a conclusion. Cite as [file name]."),
            ("human", "Question: {question}\n\nContext grouped by document:\n{context}"),
        ]) | self.llm | to_text

    # -------------------------------------------------------------- helpers
    def _retrieve(self, question, file_names=None):
        return retrieve(self.index, question, self.top_k, file_names, self.similarity_cutoff)

    def standalone_question(self, question: str) -> str:
        if not self.chat_history:
            return question
        try:
            q = self.condense_chain.invoke({"history": self.chat_history[-6:], "question": question})
            return q.strip().strip('"') or question
        except Exception:
            return question

    # -------------------------------------------------------------- public API
    def ask(self, question: str, file_names: list | None = None, use_history: bool = True) -> dict:
        """Answer a question with citations. Returns a dict with answer, sources, confidence, ..."""
        standalone = self.standalone_question(question) if use_history else question
        out = self.qa_chain.invoke({"question": standalone, "file_names": file_names})
        nodes, raw = out["nodes"], out["answer"]
        try:
            parsed = self.answer_parser.parse(raw).model_dump()
        except Exception:
            # fallback: the model did not return valid JSON -> plain answer
            text = raw if raw and not raw.lstrip().startswith("{") else self.plain_qa_chain.invoke(
                {"question": standalone, "context": format_nodes(nodes)})
            parsed = {"answer": text, "sources_used": list(range(1, len(nodes) + 1)),
                      "confidence": "medium", "follow_up_questions": []}

        self.chat_history += [HumanMessage(content=question), AIMessage(content=parsed["answer"])]
        parsed.update({"question": question, "standalone_question": standalone, "sources": nodes_to_sources(nodes)})
        return parsed

    def summarize(self, file_name: str, max_parts: int = 4, chars_per_part: int = 5000) -> str:
        """Map-reduce summary of one document (limited number of parts to respect free-tier token limits)."""
        nodes = get_document_nodes(self.index, file_name)
        if not nodes:
            return f"Document '{file_name}' not found in the index."
        text = "\n".join(n.get_content() for n in nodes)
        parts = [text[i:i + chars_per_part] for i in range(0, len(text), chars_per_part)]
        if len(parts) > max_parts:                        # spread the sample across the whole document
            step = len(parts) / max_parts
            parts = [parts[int(i * step)] for i in range(max_parts)]
        partial = [self.map_chain.invoke({"file_name": file_name, "text": p}) for p in parts]
        if len(partial) == 1:
            return partial[0]
        return self.reduce_chain.invoke({"file_name": file_name, "summaries": "\n\n".join(partial)})

    def compare(self, question: str, file_names: list, k_per_doc: int = 3) -> dict:
        """Multi-document querying: retrieve separately from each document, then compare."""
        groups, sources = [], []
        for name in file_names:
            nodes = retrieve(self.index, question, k_per_doc, [name], self.similarity_cutoff)
            groups.append(f"=== {name} ===\n" + "\n---\n".join(n.node.get_content().strip() for n in nodes))
            sources += nodes_to_sources(nodes)
        answer = self.compare_chain.invoke({"question": question, "context": "\n\n".join(groups)})
        return {"answer": answer, "sources": sources}

    def reset_memory(self):
        self.chat_history = []

## Step 4 — Ask questions

In [ ]:
from IPython.display import Markdown, display

assistant = DocumentAssistant(index, llm, top_k=4)

def show(res):
    display(Markdown(res["answer"]))
    print("Confidence:", res.get("confidence"), "| Sources used:", res.get("sources_used"))
    for s in res["sources"]:
        print(f"  [{s['n']}] {s['file_name']} p.{s['page']}  score={s['score']}")

res = assistant.ask("What is ReAct and how is it different from chain-of-thought prompting?")
show(res)

### 4.1 Context-aware follow-up (conversation memory → standalone question)

In [ ]:
res = assistant.ask("Which benchmarks did they evaluate it on?")
print("Rewritten question:", res["standalone_question"])
show(res)

### 4.2 A question that is NOT in the documents (should say it cannot find it)

In [ ]:
show(assistant.ask("What is the capital of Australia?", use_history=False))

### 4.3 Query only one document

In [ ]:
show(assistant.ask("How is hallucination detected?", file_names=["LLM_Hallucinations_Survey.pdf"], use_history=False))

## Step 5 — Summarisation (map-reduce with LangChain)

In [ ]:
display(Markdown(assistant.summarize("ReAct_Reasoning_and_Acting.pdf")))

## Step 6 — Multi-document comparison

In [ ]:
res = assistant.compare("What problem does each paper solve and what is its key idea?",
                        ["ReAct_Reasoning_and_Acting.pdf", "Attention_Is_All_You_Need.pdf"])
display(Markdown(res["answer"]))

## Conclusion & limitations
- LlamaIndex handles ingestion, chunking, embeddings, persistence and filtered retrieval; LangChain handles prompts, chains, memory and structured output.
- Answers are grounded with numbered citations and a confidence level; out-of-scope questions are refused.
- Limitations: scanned PDFs (images) need OCR; very long documents are summarised from a sample of parts to stay within free-tier token limits; small embedding model trades some accuracy for speed.
- Future work: hybrid BM25 + vector search, re-ranking, evaluation with RAGAS, LangSmith tracing.